Importing of libraries

In [4]:
pip install matplotlib seaborn scikit-learn


  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached contourpy-1.2.1-cp310-cp310-win_amd64.whl.metadata (5.8 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
     ---------------------------------------- 0.0/165.0 kB ? eta -:--:--
     -------------------------------------- 165.0/165.0 kB 3.3 MB/s eta 0:00:00
  Using cached kiwisolver-1.4.5-cp310-cp310-win_amd64.whl.metadata (6.5 kB)
  Using cached pillow-10.3.0-cp310-cp310-win_amd64.whl.metadata (9.4 kB)
  Using cached pyparsing-3.1.2-py3-none-any.whl.metadata (5.1 kB)
     ---------------------------------------- 0.0/60.6 kB ? eta -:--:--
     ---------------------------------------- 60.6/60.6 kB 3.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.0 MB 10.2 MB/s eta 0:00:01
   --- ------------------------------------ 0.7/8.0 MB 9.5 MB/s eta 0:00:01
   ----- ---------------------------------- 1.1/8.0 M

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [2]:
# Load dataset
data = pd.read_csv('C:/Users/Rajasekar/OneDrive/_12_01_DATA_ANALYST/_99_data_sets/owid-covid-data.csv')

In [3]:
# Display first few rows of the dataset
print("First 5 rows of the dataset:")
print(data.head())

# Displaying the columns of the dataset
print("All columns in the dataset:")
print(data.columns)

First 5 rows of the dataset:
  iso_code continent     location        date  population_density  \
0      AFG      Asia  Afghanistan  05-01-2020              54.422   
1      AFG      Asia  Afghanistan  06-01-2020              54.422   
2      AFG      Asia  Afghanistan  07-01-2020              54.422   
3      AFG      Asia  Afghanistan  08-01-2020              54.422   
4      AFG      Asia  Afghanistan  09-01-2020              54.422   

   total_cases  new_cases  new_cases_smoothed  total_deaths  new_deaths  ...  \
0          NaN        0.0                 NaN           NaN         0.0  ...   
1          NaN        0.0                 NaN           NaN         0.0  ...   
2          NaN        0.0                 NaN           NaN         0.0  ...   
3          NaN        0.0                 NaN           NaN         0.0  ...   
4          NaN        0.0                 NaN           NaN         0.0  ...   

   new_deaths_smoothed_per_million  reproduction_rate  icu_patients  \
0   

In [4]:
# Check for missing values
missing_values = data.isnull().sum()
print("Missing values in each column:")
print(missing_values)

Missing values in each column:
iso_code                                   0
continent                              18066
location                                   0
date                                       0
population_density                     56653
total_cases                            38922
new_cases                              10891
new_cases_smoothed                     12121
total_deaths                           60894
new_deaths                             10883
new_deaths_smoothed                    12113
total_cases_per_million                38922
new_cases_per_million                  10891
new_cases_smoothed_per_million         12121
total_deaths_per_million               60894
new_deaths_per_million                 10883
new_deaths_smoothed_per_million        12113
reproduction_rate                     192190
icu_patients                          338574
icu_patients_per_million              338574
hosp_patients                         337075
hosp_patients_per_millio

In [5]:
# Drop unwanted columns
data.drop(['continent'], axis=1, inplace=True)


In [6]:
# Imputation for missing values
num_cols = data.select_dtypes(include=['float64', 'int64']).columns
cat_cols = data.select_dtypes(include=['object']).columns

num_imputer = SimpleImputer(strategy='mean')
cat_imputer = SimpleImputer(strategy='most_frequent')

In [7]:
# Apply imputers
data[num_cols] = num_imputer.fit_transform(data[num_cols])
data[cat_cols] = cat_imputer.fit_transform(data[cat_cols])

In [ ]:
# Handle outliers
def handle_outliers(data, threshold=3):
    mean = np.mean(data)
    std_dev = np.std(data)
    outliers = (data - mean).abs() > threshold * std_dev
    cleaned_data = data.mask(outliers)
    return cleaned_data


# Select the column containing the numerical data you want to handle outliers for
# Example: replace 'your_column_name' with your actual column name
column_name = 'your_column_name'

# Handle outliers for the selected column
data[column_name] = handle_outliers(data[column_name])

In [ ]:
# Visualize the data before and after handling outliers
plt.figure(figsize=(10, 6))
plt.subplot(2, 1, 1)
plt.hist(data[column_name].dropna(), bins=20, color='blue', alpha=0.5)
plt.title('Data with Outliers')

plt.subplot(2, 1, 2)
plt.hist(data[column_name].dropna(), bins=20, color='green', alpha=0.5)
plt.title('Data after Outlier Handling')

plt.tight_layout()
plt.show()

In [ ]:
# Feature scaling
scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(data[num_cols])
scaled_df = pd.DataFrame(scaled_features, columns=num_cols)

# Standardizing a single column
scaler = StandardScaler()
data['standardized_column'] = scaler.fit_transform(data[['column_to_standardize']])

In [ ]:
# Encoding categorical variables
label_encoder = LabelEncoder()
for col in cat_cols:
    data[col] = label_encoder.fit_transform(data[col])

# OneHotEncoder for categorical variables
one_hot_encoder = OneHotEncoder(sparse=False)
encoded_features = one_hot_encoder.fit_transform(data[cat_cols])
encoded_df = pd.DataFrame(encoded_features, columns=one_hot_encoder.get_feature_names_out(cat_cols))
data = pd.concat([data.drop(columns=cat_cols), encoded_df], axis=1)

In [ ]:
# Date-time features
data['date'] = pd.to_datetime(data['date_column'])  # Convert to datetime format
data['year'] = data['date'].dt.year
data['month'] = data['date'].dt.month
data['day'] = data['date'].dt.day

In [ ]:
# Additional feature engineering
data['interaction_feature'] = data['feature1'] * data['feature2']

In [ ]:
# Split the dataset into features and target variable
X = data.drop('target_column', axis=1)
y = data['target_column']

In [ ]:
# Principal Component Analysis (PCA)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
principal_components = pca.fit_transform(X_scaled)
principal_df = pd.DataFrame(data=principal_components, columns=['PC1', 'PC2'])

explained_variance_ratio = pca.explained_variance_ratio_
print("Explained variance ratio:", explained_variance_ratio)

plt.figure(figsize=(8, 6))
plt.bar(range(len(explained_variance_ratio)), explained_variance_ratio, alpha=0.5, align='center')
plt.xlabel('Principal Components')
plt.ylabel('Explained Variance Ratio')
plt.title('Explained Variance Ratio of Principal Components')
plt.show()

plt.scatter(principal_df['PC1'], principal_df['PC2'], c=y, cmap='viridis')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('PCA Visualization')
plt.colorbar()
plt.show()

In [ ]:
# t-SNE
tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(X)

plt.figure(figsize=(10, 8))
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y, cmap='viridis')
plt.colorbar()
plt.title('t-SNE Visualization')
plt.xlabel('First t-SNE Component')
plt.ylabel('Second t-SNE Component')
plt.show()